## Development notebook

Here we check in pixel-by-pixel manner the differences between PIL and our re-implementation for drawing lines, circles etc.

In [ ]:
import sys, pathlib, os
sys.path.append(pathlib.Path.cwd().parents[1].__str__())
os.environ["IMAGE_UTILS_LOGLEVEL"] = "2"
os.environ["DEBUG_DRAW_LINE"] = "1"
from pprint import pprint
from image_utils import (image_draw_rectangle, image_draw_circle, image_draw_line, Color, PointIJ, _check_image,
                         image_draw_line, _get_px_from_perc, image_draw_polygon)
from image_utils_pil import image_draw_line_pil, image_draw_polygon_pil
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

np.set_printoptions(linewidth=200)

%reload_ext autoreload
%autoreload 2

In [ ]:
def white(*size) -> np.ndarray:
    return (np.ones((*size, 3)) * Color.WHITE).astype(np.uint8)

def black(*size) -> np.ndarray:
    return (np.ones((*size, 3)) * Color.BLACK).astype(np.uint8)

def print_list(lst: list[int]):
    print("[", end="")
    for i, row in enumerate(lst):
        print("[", end="")
        print(", ".join([f"{x:3d}" for x in row]), end="")
        if i < len(lst) - 1:
            print("],\n ", end="")
        else:
            print("]", end="")
    print("]")

def image_display(image: np.ndarray):
    display(Image.fromarray(image)) # pylint: disable=all # noqa: F821


## Drawing lines

Turns out drawing lines is hard. The `size, p1, p2, thickness` tuple is used as-is in `image_draw_vs_pil_test.py` for the parametrized tests.

In [ ]:

size, p1, p2, thickness = (10, 10), (2, 2), (6, 6), 30
# p1, p2 = p2, p1
# size = (20, 20)

thickness = thickness
img = white(*size)

# image_display(res_pil)
points = [p1, p2]
# points = [(10, 15), (15, 20)]#, (20, 10)]

res_pil = image_draw_line_pil(img, p1=p1, p2=p2, color=Color.BLACK, thickness=thickness)
# res_pil = image_draw_polygon_pil(img, points, color=Color.BLACK, thickness=thickness)
print_list(res_pil[..., 0].tolist())
# image_display(res_pil)

# res_pil = image_draw_polygon_pil(img, points, color=Color.BLACK, thickness=thickness)
# image_display(res_pil)

print()
res2 = image_draw_line(img, p1=p1, p2=p2, color=Color.BLACK, thickness=thickness)
# res2 = image_draw_polygon(img, points, color=Color.BLACK, thickness=thickness)
# image_display(res2)
print_list(res2[..., 0].tolist())

res2[(res2!=0) & (res2!=99)] = 255 # for debug (with 11, 22 etc.)
res2[(res2==0) & (res_pil == 255)] = 255 # fill in bottom part for debug
close = (mean := (res_pil != res2).mean()) <= 0.048
print(f"Equals: {np.allclose(res_pil, res2)}. Mean: {round(mean, 5)} (close={close.item()}).")

In [ ]:
img = white(250, 250)
points = [(100, 100), (100, 150)]
# points = [(25, 25), (25, 75)]
res_pil = image_draw_polygon_pil(img, points, color=(0, 0, 255), thickness=1)
res = image_draw_polygon(img, points, color=(0, 0, 255), thickness=1)
image_display(res_pil)
image_display(res)
print(np.allclose(res_pil, res))